# Лабораторная работа 1 — рабочий ноутбук

**ФИО:**
**Файлы данных:**

Этот ноутбук — чистовой рабочий документ, а не черновик. Он должен
исполняться сверху вниз без ошибок на выданном вам **сыром** файле
и на выходе давать очищенную таблицу и `cleaning_log.csv`.

Разделы ниже обязательны, порядок менять не нужно. Внутри раздела
добавляйте столько ячеек, сколько нужно. Если какой-то раздел у вас
пустой — напишите в нём одной строкой, почему.

## 0. Окружение и настройки

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

# Пути к файлам вашего варианта S32
RAW_CLIENTS = os.path.join("data", "S32", "clients.csv")
RAW_TXN = os.path.join("data", "S32", "transactions.csv")

## 1. Загрузка сырых данных

Файлы выгружены разными системами. Убедитесь, что они прочитались
**правильно**: кодировка, разделитель, десятичный разделитель, типы,
даты, идентификаторы.

In [3]:
import csv

def detect_file_params(filepath):
    """Определяет кодировку и разделитель файла."""
    # 1. Проверяем популярные кодировки
    encodings = ['utf-8', 'windows-1251', 'cp1251', 'utf-8-sig']
    correct_enc = None
    sample_text = ""
    
    for enc in encodings:
        try:
            with open(filepath, 'r', encoding=enc) as f:
                sample_text = "".join([f.readline() for _ in range(15)])
                correct_enc = enc
                break
        except (UnicodeDecodeError, UnicodeError):
            continue
            
    # 2. Определяем разделитель колонок
    sniffer = csv.Sniffer()
    try:
        dialect = sniffer.sniff(sample_text)
        delimiter = dialect.delimiter
    except Exception:
        # Резервная эвристика, если sniffer не справился
        first_line = sample_text.split('\n')[0]
        delimiter = ';' if ';' in first_line else (',' if ',' in first_line else '\t')
        
    return correct_enc, delimiter

# Исследуем параметры файлов
clients_enc, clients_sep = detect_file_params(RAW_CLIENTS)
txn_enc, txn_sep = detect_file_params(RAW_TXN)

print(f"Параметры {RAW_CLIENTS}: кодировка={clients_enc}, разделитель='{clients_sep}'")
print(f"Параметры {RAW_TXN}: кодировка={txn_enc}, разделитель='{txn_sep}'")

# Загружаем сырые таблицы
# Пока не форсируем типы и парсинг дат, чтобы в разделе аудита увидеть исходные данные
df_clients_raw = pd.read_csv(RAW_CLIENTS, encoding=clients_enc, sep=clients_sep)
df_txn_raw = pd.read_csv(RAW_TXN, encoding=txn_enc, sep=txn_sep)

print("\n=== clients.csv: первые 3 строки ===")
display(df_clients_raw.head(3))
print(f"Размер: {df_clients_raw.shape[0]} строк, {df_clients_raw.shape[1]} колонок")

print("\n=== transactions.csv: первые 3 строки ===")
display(df_txn_raw.head(3))
print(f"Размер: {df_txn_raw.shape[0]} строк, {df_txn_raw.shape[1]} колонок")

Параметры data\S32\clients.csv: кодировка=utf-8, разделитель=','
Параметры data\S32\transactions.csv: кодировка=windows-1251, разделитель=';'

=== clients.csv: первые 3 строки ===


,client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in
0,C000864,1959-02-13,М,Нижний Новгород,Массовый,Серебро,Дебетовая,11.02.2021,29.07.2024,Не указан,37200.0,Да,2024-12,NaN,300,Нет
1,C000608,1980-05-27,М,Санкт-Петербург,Массовый,Серебро,Дебетовая,08.06.2012,19.05.2018,Не указан,51400.0,Да,2024-06,NaN,131,Да
2,C005220,1963-07-31,М,Санкт Петербург,Зарплатный,Золото,Дебетовая,15.02.2023,26.07.2024,Зарплата в нашем банке,115200.0,Да,2025-03,5.0,845,Нет


Размер: 5693 строк, 16 колонок

=== transactions.csv: первые 3 строки ===


,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
0,990147,C001784,2023-12-01 05:36:36,"439,6",RUB,Связь и интернет,4814,СБП,успешно
1,990049,C002958,2023-12-04 11:32:23,"48461,62",RUB,Одежда и обувь,5651,Карта,успешно
2,990214,C002389,2023-12-10 05:52:56,"763,12",RUB,Связь и Интернет,4814,Онлайн-банк,успешно


Размер: 628754 строк, 9 колонок


## 2. Аудит качества

Здесь вы ничего не исправляете — только находите и измеряете.
Для каждой метрики качества заранее определите знаменатель.

### 2.1. Объём, типы, ключи, единица наблюдения

In [4]:
print("=== 2.1. ОБЪЁМ И СТРУКТУРА ДАННЫХ ===")
print(f"Клиенты (clients.csv): {df_clients_raw.shape[0]} строк, {df_clients_raw.shape[1]} колонок")
print(f"Транзакции (transactions.csv): {df_txn_raw.shape[0]} строк, {df_txn_raw.shape[1]} колонок\n")

print("--- Типы данных clients.csv ---")
print(df_clients_raw.dtypes)

print("\n--- Типы данных transactions.csv ---")
print(df_txn_raw.dtypes)

# Проверка уникальности первичных ключей (Primary Keys)
client_pk_unique = df_clients_raw['client_id'].nunique()
client_pk_total = len(df_clients_raw)
print(f"\nУникальность client_id в clients: {client_pk_unique} из {client_pk_total} (дублей: {client_pk_total - client_pk_unique})")

txn_pk_unique = df_txn_raw['txn_id'].nunique()
txn_pk_total = len(df_txn_raw)
print(f"Уникальность txn_id в transactions: {txn_pk_unique} из {txn_pk_total} (дублей: {txn_pk_total - txn_pk_unique})")

print("\nЕдиница наблюдения:")
print("- Таблица clients: 1 строка = 1 уникальный действующий клиент банка на 30.06.2025")
print("- Таблица transactions: 1 строка = 1 операция по банковской карте")

=== 2.1. ОБЪЁМ И СТРУКТУРА ДАННЫХ ===
Клиенты (clients.csv): 5693 строк, 16 колонок
Транзакции (transactions.csv): 628754 строк, 9 колонок

--- Типы данных clients.csv ---
client_id              object
birth_date             object
gender                 object
city                   object
segment                object
loyalty_tier           object
card_type              object
registration_date      object
card_open_date         object
income_source          object
monthly_income        float64
has_mobile_app         object
app_last_login         object
satisfaction_score    float64
branch_code             int64
marketing_opt_in       object
dtype: object

--- Типы данных transactions.csv ---
txn_id           int64
client_id       object
txn_datetime    object
amount          object
currency        object
category        object
mcc_code         int64
channel         object
status          object
dtype: object

Уникальность client_id в clients: 5638 из 5693 (дублей: 55)
Уникальность t

### 2.2. Дубли

In [5]:
print("=== 2.2. АУДИТ ДУБЛИКАТОВ ===")

# 1. Полные дубликаты строк
full_dup_clients = df_clients_raw.duplicated().sum()
full_dup_txn = df_txn_raw.duplicated().sum()

print(f"Полные дубли в clients: {full_dup_clients} из {len(df_clients_raw)} строк ({full_dup_clients/len(df_clients_raw):.4%})")
print(f"Полные дубли в transactions: {full_dup_txn} из {len(df_txn_raw)} строк ({full_dup_txn/len(df_txn_raw):.4%})")

# 2. Дубликаты по ключевым полям (ID)
key_dup_clients = df_clients_raw.duplicated(subset=['client_id']).sum()
key_dup_txn = df_txn_raw.duplicated(subset=['txn_id']).sum()

print(f"\nДубли по client_id в clients: {key_dup_clients} из {len(df_clients_raw)} ({key_dup_clients/len(df_clients_raw):.4%})")
print(f"Дубли по txn_id в transactions: {key_dup_txn} из {len(df_txn_raw)} ({key_dup_txn/len(df_txn_raw):.4%})")

=== 2.2. АУДИТ ДУБЛИКАТОВ ===
Полные дубли в clients: 42 из 5693 строк (0.7377%)
Полные дубли в transactions: 1279 из 628754 строк (0.2034%)

Дубли по client_id в clients: 55 из 5693 (0.9661%)
Дубли по txn_id в transactions: 1500 из 628754 (0.2386%)


### 2.3. Соответствие данных описанию полей из задания

In [6]:
print("=== 2.3. СООТВЕТСТВИЕ ОПИСАНИЮ ПОЛЕЙ ===\n")

print("--- 1. Категориальные поля clients ---")
for col in ['gender', 'segment', 'loyalty_tier', 'card_type', 'has_mobile_app', 'marketing_opt_in']:
    unique_vals = df_clients_raw[col].dropna().unique().tolist()
    print(f"{col} (уникальных: {len(unique_vals)}): {unique_vals}")

print("\n--- 2. Города (первые 20 и общее число) ---")
cities = df_clients_raw['city'].dropna().unique().tolist()
print(f"Всего уникальных написаний городов: {len(cities)}")
print("Примеры городов:", sorted(cities)[:15])

print("\n--- 3. Числовые поля clients ---")
print("satisfaction_score (уникальные значения):", sorted(df_clients_raw['satisfaction_score'].unique().tolist()))
print(f"monthly_income: min = {df_clients_raw['monthly_income'].min()}, max = {df_clients_raw['monthly_income'].max()}")

print("\n--- 4. Категориальные поля transactions ---")
for col in ['currency', 'channel', 'status']:
    print(f"{col}: {df_txn_raw[col].dropna().unique().tolist()}")

print("\nКатегории трат (category):")
categories = df_txn_raw['category'].dropna().unique().tolist()
print(f"Всего вариантов написания категорий: {len(categories)} (по ТЗ должно быть ровно 9)")
print("Варианты:", categories)

print("\n--- 5. Диапазон дат в транзакциях ---")
# Временный срез первых 10 символов (YYYY-MM-DD) для оценки диапазона
dates_sample = df_txn_raw['txn_datetime'].str[:10]
min_date, max_date = dates_sample.min(), dates_sample.max()
print(f"Фактический диапазон дат в transactions: от {min_date} до {max_date}")
print(f"Ожидаемый диапазон по ТЗ: от 2024-01-01 до 2025-06-30")

# Сколько транзакций выходят за рамки ТЗ
out_of_bounds = (~dates_sample.between("2024-01-01", "2025-06-30")).sum()
print(f"Транзакций вне периода исследования: {out_of_bounds} из {len(df_txn_raw)} ({out_of_bounds/len(df_txn_raw):.4%})")

=== 2.3. СООТВЕТСТВИЕ ОПИСАНИЮ ПОЛЕЙ ===

--- 1. Категориальные поля clients ---
gender (уникальных: 2): ['М', 'Ж']
segment (уникальных: 4): ['Массовый', 'Зарплатный', 'Премиальный', 'Private']
loyalty_tier (уникальных: 4): ['Серебро', 'Золото', 'Базовый', 'Платина']
card_type (уникальных: 3): ['Дебетовая', 'Кредитная', 'Премиальная']
has_mobile_app (уникальных: 2): ['Да', 'Нет']
marketing_opt_in (уникальных: 2): ['Нет', 'Да']

--- 2. Города (первые 20 и общее число) ---
Всего уникальных написаний городов: 17
Примеры городов: [' москва', 'ЕКАТЕРИНБУРГ', 'Екатеринбург', 'Казань', 'МОСКВА', 'Москва', 'Н. Новгород', 'Нижний Новгород', 'Новосибирск', 'СПб', 'Санкт Петербург', 'Санкт-Петербург', 'г. Москва', 'г. Новосибирск', 'г. Санкт-Петербург']

--- 3. Числовые поля clients ---
satisfaction_score (уникальные значения): [nan, 1.0, 2.0, 3.0, 4.0, 5.0]
monthly_income: min = -999.0, max = 1026300.0

--- 4. Категориальные поля transactions ---
currency: ['RUB', 'USD', 'EUR']
channel: ['СБП', 

### 2.4. Связь между таблицами

In [7]:
print("=== 2.4. СВЯЗЬ МЕЖДУ ТАБЛИЦАМИ (РЕФЕРЕНЦИАЛЬНАЯ ЦЕЛОСТНОСТЬ) ===")

set_clients_in_dict = set(df_clients_raw['client_id'])
set_clients_in_txn = set(df_txn_raw['client_id'])

# 1. Клиенты, у которых есть транзакции, но их нет в справочнике клиентов
orphan_txns_clients = set_clients_in_txn - set_clients_in_dict
orphan_txns_count = df_txn_raw['client_id'].isin(orphan_txns_clients).sum()
print(f"Транзакций от 'клиентов-призраков' (нет в clients.csv): {orphan_txns_count} из {len(df_txn_raw)} ({orphan_txns_count/len(df_txn_raw):.4%})")
print(f"Уникальных 'клиентов-призраков': {len(orphan_txns_clients)}")

# 2. Клиенты из справочника, у которых ВООБЩЕ нет ни одной транзакции
clients_without_txns = set_clients_in_dict - set_clients_in_txn
print(f"\nКлиентов без единой транзакции: {len(clients_without_txns)} из {len(df_clients_raw)} ({len(clients_without_txns)/len(df_clients_raw):.4%})")

=== 2.4. СВЯЗЬ МЕЖДУ ТАБЛИЦАМИ (РЕФЕРЕНЦИАЛЬНАЯ ЦЕЛОСТНОСТЬ) ===
Транзакций от 'клиентов-призраков' (нет в clients.csv): 30810 из 628754 (4.9002%)
Уникальных 'клиентов-призраков': 362

Клиентов без единой транзакции: 11 из 5693 (0.1932%)


**Выводы по аудиту.** Перечислите найденное: что это, сколько строк,
насколько это мешает решить задачу.

## 3. Препроцессинг

На каждое изменение — строка в журнале (раздел 4). Решение «оставить
как есть» тоже является решением и тоже попадает в журнал.

### 3.1. Типы, даты, числа

### 3.2. Категориальные признаки

### 3.3. Пропуски

Для каждого поля с пропусками: доля, предположение о механизме, решение.

### 3.4. Выбросы

Обнаружение и решение — это два разных шага. Сначала первый, потом второй.

### 3.5. Фильтры, применяемые перед расчётом активности

Покажите воронку: сколько строк на входе и сколько осталось после каждого фильтра.

## 4. Журнал изменений

Одна строка — одно решение, а не одна запись. Колонки: `field`, `issue`,
`scope`, `n_rows`, `value_before`, `action`, `value_after`, `rationale`.

In [ ]:
cleaning_log = pd.DataFrame(columns=[
    "field", "issue", "scope", "n_rows",
    "value_before", "action", "value_after", "rationale",
])

## 5. Разведочный анализ

Графиков столько, сколько нужно. В отчёт пойдут не все.

### 5.1. Распределения количественных признаков

### 5.2. Частотные характеристики категориальных признаков

### 5.3. Динамика по месяцам

### 5.4. Разрезы и связи между признаками

## 6. Метрика активности

Что именно вы считаете активностью, как она вычисляется, на каком
множестве клиентов, за какие окна. Отдельно: какое снижение эта метрика
**не увидит**.

## 7. Поиск и описание групп

Правило отбора для каждой группы, её размер, причина падения.
Правило разрешения конфликтов, если клиент подходит под несколько групп.

### 7.1. Кого в кампанию не берём и почему

## 8. Что проверили и что не подтвердилось

Гипотезы, которые вы проверили и отвергли. По одной строке на гипотезу,
с числом.

## 9. Сохранение результатов

Три файла: очищенная таблица для второго ноутбука, журнал изменений,
итоговый список кампании.

In [ ]:
# clean.to_parquet("clean_data.parquet", index=False)
# cleaning_log.to_csv("cleaning_log.csv", index=False, encoding="utf-8")
# campaign[["client_id", "group"]].to_csv("campaign_list.csv", index=False, encoding="utf-8")